# Runbook 11 — Kerberos Client Authentication Integration

> **KDC:** `kerberos-kdc.prod.svc.cluster.local:88` · **Realm:** `STARDATADBLABS.LOCAL`  
> **Source doc:** [runbook-11-kerberos-integration.md](runbook-11-kerberos-integration.md)  
> **Related runbooks:** [01 — OpenBao](runbook-01-openbao.md) · [08 — Security & Access](runbook-08-security-access.md)

Every `bash` cell runs via the `!` shell escape. All cells require `kubectl` on `$PATH`
and a valid `kubeconfig` pointing at the `prod` cluster.

**Set the variables in §0 before running anything else.**

---
## § 0 — Global Variables

Edit these once; every subsequent cell references them.

In [ ]:
import os, json, subprocess

# ── Cluster / namespace ──────────────────────────────────────────────────────
NAMESPACE   = "prod"
REALM       = "STARDATADBLABS.LOCAL"

# ── OpenBao ──────────────────────────────────────────────────────────────────
BAO_ADDR    = "http://192.168.1.50:30820"
KEYS_FILE   = os.path.expanduser("~/openbao-init-keys.json")
if not os.path.exists(KEYS_FILE):
    KEYS_FILE = "/root/openbao-init-keys.json"

with open(KEYS_FILE) as fh:
    ROOT_TOKEN = json.load(fh)["root_token"]

print(f"BAO_ADDR   : {BAO_ADDR}")
print(f"ROOT_TOKEN : {ROOT_TOKEN[:8]}… (masked)")
print(f"NAMESPACE  : {NAMESPACE}")

---
## § 1 — Platform Toggle

In [ ]:
# Check current Kerberos toggle state
!kubectl get cm kerberos-integration-config -n {NAMESPACE} \
  -o jsonpath='{{.data.kerberos\.enabled}}' && echo

In [ ]:
# ── Enable Kerberos platform-wide ─────────────────────────────────────────────
# Step 1 — flip the toggle
!kubectl patch cm kerberos-integration-config -n {NAMESPACE} \
  --type merge -p '{{"data":{{"kerberos.enabled":"true"}}}}'

# Step 2 — restart Spark (Kafka and OpenSearch need manual manifest changes — see §5/§6)
!kubectl rollout restart deploy/spark-master deploy/spark-worker -n {NAMESPACE}
!kubectl rollout restart statefulset/doris-fe -n {NAMESPACE}

In [ ]:
# ── Disable Kerberos platform-wide ────────────────────────────────────────────
# Uncomment to run
# !kubectl patch cm kerberos-integration-config -n {NAMESPACE} \
#   --type merge -p '{"data":{"kerberos.enabled":"false"}}'
# !kubectl rollout restart deploy/spark-master deploy/spark-worker -n {NAMESPACE}
# !kubectl rollout restart statefulset/doris-fe -n {NAMESPACE}

---
## § 2 — Verify Service Keytabs

In [ ]:
import subprocess, tempfile, os

def verify_keytab(secret_name: str, ns: str = NAMESPACE) -> None:
    """Extract a keytab K8s secret to a temp file and run klist -ekt."""
    result = subprocess.run(
        ["kubectl", "get", "secret", secret_name, "-n", ns,
         "-o", "jsonpath={.data.keytab}"],
        capture_output=True, text=True, check=True
    )
    import base64
    keytab_bytes = base64.b64decode(result.stdout.strip())
    with tempfile.NamedTemporaryFile(delete=False, suffix=".keytab") as tmp:
        tmp.write(keytab_bytes)
        tmp_path = tmp.name
    try:
        out = subprocess.run(["klist", "-ekt", tmp_path],
                             capture_output=True, text=True)
        print(f"=== {secret_name} ===")
        print(out.stdout or out.stderr)
    finally:
        os.unlink(tmp_path)

for svc_secret in ["kafka-keytab", "doris-keytab", "spark-keytab", "opensearch-keytab"]:
    verify_keytab(svc_secret)

---
## § 3 — Add a New User

> Set `NEW_USER` and `NEW_USER_PASS` (Doris SQL password) before running the cells below.

In [ ]:
NEW_USER      = "alice"              # ← change me
NEW_USER_PASS = "AliceDoris1!"       # ← Doris SQL password (not the KDC password)
KDC_TMP_PASS  = "TempPass1!"         # ← temporary KDC password (user should change)

In [ ]:
# Step 1 — Create the KDC principal (non-interactive)
!kubectl exec -n {NAMESPACE} deploy/kerberos-kdc -- \
  kadmin.local -q "addprinc -pw {KDC_TMP_PASS} {NEW_USER}@{REALM}"

# Verify
!kubectl exec -n {NAMESPACE} deploy/kerberos-kdc -- \
  kadmin.local -q "getprinc {NEW_USER}@{REALM}"

In [ ]:
# Step 2 — Export keytab and store as K8s secret
import subprocess, base64, tempfile, os

# Export keytab inside KDC pod
!kubectl exec -n {NAMESPACE} deploy/kerberos-kdc -- \
  kadmin.local -q "ktadd -k /tmp/{NEW_USER}.keytab {NEW_USER}@{REALM}"

# Get KDC pod name and copy keytab out
kdc_pod = subprocess.check_output(
    ["kubectl", "get", "pod", "-n", NAMESPACE, "-l", "app=kerberos-kdc",
     "-o", "jsonpath={.items[0].metadata.name}"],
    text=True
).strip()
print(f"KDC pod: {kdc_pod}")

keytab_local = f"/tmp/{NEW_USER}.keytab"
!kubectl cp {NAMESPACE}/{kdc_pod}:/tmp/{NEW_USER}.keytab {keytab_local}

# Verify keytab
!klist -ekt {keytab_local}

# Store as K8s secret
!kubectl create secret generic {NEW_USER}-keytab \
  --from-file=keytab={keytab_local} \
  -n {NAMESPACE} \
  --dry-run=client -o yaml | kubectl apply -f -

# Clean up temp files
!kubectl exec -n {NAMESPACE} deploy/kerberos-kdc -- rm /tmp/{NEW_USER}.keytab
os.unlink(keytab_local)
print("Keytab stored and temp files cleaned up.")

In [ ]:
# Step 3 — Create Doris SQL user (same short username)
import requests

resp = requests.get(
    f"{BAO_ADDR}/v1/secret/data/doris/credentials",
    headers={"X-Vault-Token": ROOT_TOKEN},
    timeout=10
)
resp.raise_for_status()
DORIS_ROOT_PASS = resp.json()["data"]["data"]["admin-password"]
print("Doris root password retrieved.")

!kubectl exec -n {NAMESPACE} statefulset/doris-fe -it -- \
  mysql -h127.0.0.1 -P9030 -uroot -p"{DORIS_ROOT_PASS}" \
  -e "CREATE USER '{NEW_USER}'@'%' IDENTIFIED BY '{NEW_USER_PASS}';"

In [ ]:
# Step 4 — Store credentials in OpenBao
import requests

headers = {"X-Vault-Token": ROOT_TOKEN, "Content-Type": "application/json"}

# Kerberos principal metadata
r1 = requests.post(
    f"{BAO_ADDR}/v1/secret/data/kerberos/users/{NEW_USER}",
    headers=headers,
    json={"data": {
        "principal":     f"{NEW_USER}@{REALM}",
        "keytab_secret": f"{NEW_USER}-keytab",
        "created_by":    "admin"
    }},
    timeout=10
)
r1.raise_for_status()
print("Kerberos entry stored")

# Doris password
r2 = requests.post(
    f"{BAO_ADDR}/v1/secret/data/doris/users/{NEW_USER}",
    headers=headers,
    json={"data": {
        "username":   NEW_USER,
        "password":   NEW_USER_PASS,
        "service":    "doris",
        "created_by": "admin"
    }},
    timeout=10
)
r2.raise_for_status()
print("Doris credential stored")

In [ ]:
# Step 5 — Grant Doris SELECT privileges
!kubectl exec -n {NAMESPACE} statefulset/doris-fe -it -- \
  mysql -h127.0.0.1 -P9030 -uroot -p"{DORIS_ROOT_PASS}" \
  -e "GRANT SELECT ON *.* TO '{NEW_USER}'@'%';"

---
## § 4 — Kafka GSSAPI Listener

> The KRB listener is **commented out by default** in `manifests/strimzi/kafka-cluster.yaml`.
> Enabling it requires a manifest edit + git commit. The cells below cover verification only.

In [ ]:
# Check Kafka broker logs for KRB / GSSAPI / port 9093 activity
!kubectl logs -n {NAMESPACE} strimzi-kafka-combined-0 \
  | grep -iE "krb|GSSAPI|9093" | tail -20

In [ ]:
# Write a GSSAPI client.properties for manual testing (requires kinit first)
krb_props = """\
security.protocol=SASL_PLAINTEXT
sasl.mechanism=GSSAPI
sasl.kerberos.service.name=svc
sasl.jaas.config=com.sun.security.auth.module.Krb5LoginModule required \\
  useTicketCache=true;
"""

with open("/tmp/krb-client.properties", "w") as fh:
    fh.write(krb_props)
print("Written: /tmp/krb-client.properties")
print(krb_props)

---
## § 5 — OpenSearch SPNEGO

> Enable by changing `kerberos_auth_domain.http_enabled: false → true` in
> `manifests/opensearch/opensearch-security-config.yaml`, then run the cells below.

In [ ]:
# Apply updated security config.yml via securityadmin.sh
SECURITY_CONFIG_YAML = "manifests/opensearch/opensearch-security-config.yaml"

!kubectl cp {SECURITY_CONFIG_YAML} \
  {NAMESPACE}/opensearch-cluster-master-0:/tmp/security-config.yaml

!kubectl exec -n {NAMESPACE} opensearch-cluster-master-0 -- bash -c "
  python3 -c \"
import yaml, sys
d = yaml.safe_load(open('/tmp/security-config.yaml'))
print(d['data']['config.yml'])
\" > /tmp/config.yml
  /usr/share/opensearch/plugins/opensearch-security/tools/securityadmin.sh \\
    -f /tmp/config.yml -t config -icl -nhnv \\
    -cacert /usr/share/opensearch/config/tls/root-ca.pem \\
    -cert   /usr/share/opensearch/config/tls/admin.pem \\
    -key    /usr/share/opensearch/config/tls/admin-key.pem \\
    -h opensearch-cluster-master-0.prod.svc.cluster.local
"

In [ ]:
# Test SPNEGO (requires a valid TGT — run kinit alice@STARDATADBLABS.LOCAL first)
import requests

# requests-kerberos provides SPNEGO negotiation
try:
    from requests_kerberos import HTTPKerberosAuth, OPTIONAL
    r = requests.get(
        "http://192.168.1.53:30920/_cluster/health",
        auth=HTTPKerberosAuth(mutual_authentication=OPTIONAL),
        timeout=10
    )
    r.raise_for_status()
    print(r.json())
except ImportError:
    print("requests-kerberos not installed — falling back to curl:")
    !curl --negotiate -u : http://192.168.1.53:30920/_cluster/health

---
## § 6 — Spark Kerberos Job Submission

In [ ]:
# Submit a Spark job with Kerberos credentials
# Edit APP_JAR and APP_CLASS for your actual job
APP_JAR   = "/path/to/app.jar"       # ← change me
APP_CLASS = "org.example.MyApp"      # ← change me

!kubectl exec -n {NAMESPACE} deploy/spark-master -- \
  /opt/spark/bin/spark-submit \
    --master spark://spark-master-svc.{NAMESPACE}.svc.cluster.local:7077 \
    --conf spark.kerberos.enabled=true \
    --conf spark.kerberos.principal={NEW_USER}@{REALM} \
    --conf spark.kerberos.keytab=/etc/security/keytabs/{NEW_USER}.keytab \
    --class {APP_CLASS} \
    {APP_JAR}

---
## § 7 — Remove a User

In [ ]:
REMOVE_USER = "alice"   # ← change me — safety: explicit variable, not NEW_USER

In [ ]:
# 1. Delete KDC principal
!kubectl exec -n {NAMESPACE} deploy/kerberos-kdc -- \
  kadmin.local -q "delprinc -force {REMOVE_USER}@{REALM}"

# 2. Delete K8s keytab secret
!kubectl delete secret {REMOVE_USER}-keytab -n {NAMESPACE} 2>/dev/null || echo "not found"

In [ ]:
# 3. Delete Doris SQL user
import requests

resp = requests.get(
    f"{BAO_ADDR}/v1/secret/data/doris/credentials",
    headers={"X-Vault-Token": ROOT_TOKEN},
    timeout=10
)
resp.raise_for_status()
DORIS_ROOT_PASS = resp.json()["data"]["data"]["admin-password"]

!kubectl exec -n {NAMESPACE} statefulset/doris-fe -it -- \
  mysql -h127.0.0.1 -P9030 -uroot -p"{DORIS_ROOT_PASS}" \
  -e "DROP USER '{REMOVE_USER}'@'%';"

In [ ]:
# 4. Delete OpenBao entries
import requests

headers = {"X-Vault-Token": ROOT_TOKEN}

r1 = requests.delete(
    f"{BAO_ADDR}/v1/secret/data/kerberos/users/{REMOVE_USER}",
    headers=headers, timeout=10
)
print("kerberos entry deleted" if r1.ok else f"kerberos delete failed: {r1.status_code}")

r2 = requests.delete(
    f"{BAO_ADDR}/v1/secret/data/doris/users/{REMOVE_USER}",
    headers=headers, timeout=10
)
print("doris entry deleted" if r2.ok else f"doris delete failed: {r2.status_code}")

In [ ]:
# 5. Verify KDC principal is gone
result = !kubectl exec -n {NAMESPACE} deploy/kerberos-kdc -- \
  kadmin.local -q "listprincs" 2>&1
found = any(f"{REMOVE_USER}@" in line for line in result)
if found:
    print(f"WARNING: {REMOVE_USER} still in KDC!")
else:
    print(f"OK — KDC clean, {REMOVE_USER} not found")

---
## § 8 — Debugging

In [ ]:
# List all KDC principals
!kubectl exec -n {NAMESPACE} deploy/kerberos-kdc -- kadmin.local -q "listprincs"

In [ ]:
# Verify a keytab secret is intact (should show "Kerberos Keytab")
!kubectl get secret kafka-keytab -n {NAMESPACE} \
  -o jsonpath='{{.data.keytab}}' | base64 -d | file -

In [ ]:
# Check krb5.conf is mounted in service pods
!kubectl exec -n {NAMESPACE} strimzi-kafka-combined-0 -- \
  cat /mnt/krb5-conf/cluster.conf

!kubectl exec -n {NAMESPACE} statefulset/doris-fe -- \
  cat /etc/krb5.conf.d/cluster.conf 2>/dev/null || echo "not mounted"

!kubectl exec -n {NAMESPACE} deploy/spark-master -- \
  cat /etc/krb5.conf.d/cluster.conf

!kubectl exec -n {NAMESPACE} opensearch-cluster-master-0 -- \
  cat /etc/krb5.conf.d/cluster.conf

In [ ]:
# Check KDC is reachable from Spark master
!kubectl exec -n {NAMESPACE} deploy/spark-master -- \
  nc -zv kerberos-kdc.{NAMESPACE}.svc.cluster.local 88 2>&1

In [ ]:
# Check Kafka broker JVM krb5 flag
!kubectl logs -n {NAMESPACE} strimzi-kafka-combined-0 | grep "krb5" | head -5